## Imports from activity 6: apis_blank
import pandas as pd
import numpy as np
import re
import requests
import yaml

## Imports for wikipedia pulls
import pywikibot as pw
import time
pw.config.user_agent = 'EskildsenLogan research project (logan.j.eskildsen.28@dartmouth.edu) Pywikibot/11.6.0'

## imports for test visualization, if needed
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.optimize import curve_fit

## set font parameters, if needed
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman"]


## repeated printouts
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [5]:
## Creates foundational dataframe for project, scraped from 

## List of countries U.S. has played since 2014. Include year, match outcome, match score.
country_list = ["Ghana", "Portugal", "Germany", "Belgium",
                "Wales", "England", "Iran", "Netherlands", 
                "Paraguay", "Australia", "Turkey", "Bosnia_and_Herzegovina", "Belgium"]

year_played = [2014, 2014, 2014, 2014,
               2022, 2022, 2022, 2022, 
               2026, 2026, 2026, 2026, 2026]

match_outcome = ["win", "draw", "loss", "loss",
                 "draw", "draw", "win", "loss", 
                 "win", "win", "loss", "win", "loss"]

match_score = ["2-1", "2-2", "0-1", "1-2",
              "1-1", "0-0", "1-0", "1-3",
              "4-1", "2-0", "2-3", "2-0", "1-4"]

## Create simple dataframe for visualization

basic_info = pd.DataFrame({"country": country_list, "year played": year_played, "outcome": match_outcome, "score": match_score})
basic_info

,country,year played,outcome,score
0,Ghana,2014,win,2-1
1,Portugal,2014,draw,2-2
2,Germany,2014,loss,0-1
3,Belgium,2014,loss,1-2
4,Wales,2022,draw,1-1
5,England,2022,draw,0-0
6,Iran,2022,win,1-0
7,Netherlands,2022,loss,1-3
8,Paraguay,2026,win,4-1
9,Australia,2026,win,2-0


In [20]:
## Use pywikibot library to extract main Wiki subcategories for each country category. List of countries defined in the dataframe

site = pw.Site("en", "wikipedia")

## Empty list to store data later.

subcat_data = []

## User-defined function that iterates over the country column in basic_info df, gets all data on subcategories for each country.
def get_wiki_subcategories(countries):
    ## Get each category associated with country, perform a recursive loop
    for cat_name in countries:
        
        cat = pw.Category(site, cat_name)

        # Use try-except to get information on pages per subcategory
        for subcat in cat.subcategories():

            try:

                all_info = subcat.categoryinfo

                num_articles = all_info.get('pages', 0)

                cat_title = subcat.title()

                subcat_data.append({
                    "country": cat_name,
                    "subcategory_title": cat_title,
                    "article_count": num_articles
                })
                ## Need time.sleep() because I kept getting 429 errors, making too many requests.
                time.sleep(2)

            except KeyError:

                 print(f"{subcat.title()}: 0 articles (or info unavailable)")

## call function, use to_list() method
get_wiki_subcategories(basic_info["country"])

subcat_df = pd.DataFrame(subcat_data)

subcat_df

,country,subcategory_title,article_count
0,Ghana,Category:Ghana-related lists,8
1,Ghana,Category:Buildings and structures in Ghana,4
2,Ghana,Category:Culture of Ghana,56
3,Ghana,Category:Economy of Ghana,26
4,Ghana,Category:Education in Ghana,38
...,...,...,...
222,Belgium,Category:Belgian people,2
223,Belgium,Category:Politics of Belgium,40
224,Belgium,Category:Society of Belgium,12
225,Belgium,Category:Images of Belgium,1


In [43]:
## Removes stubs from subcategories since it confounds data
subcat_df_final = subcat_df[~subcat_df['subcategory_title'].str.contains('stubs', na=False)].copy()

## Now eliminate duplicates from repeat matches (Belgium)
subcat_df_final = subcat_df_final.drop_duplicates().copy()

## Format the data better by removing "Category:" using the regex skills we learned in class
subcat_df_final["subcategory_title_fixed"] = subcat_df_final["subcategory_title"].str.replace('^Category:', '', regex=True)

## Now sort by top 4 article count by subcategory for each country
subcat_df_final = (
    subcat_df_final.sort_values(by=["country", "article_count"], ascending=[True, False])
    .groupby('country')
    .head(4)
    .reset_index(drop=True)
    .copy()
)

## Now display presentable dataframe
subcat_df_final[["country", "subcategory_title_fixed", "subcategory_title", "article_count"]]

,country,subcategory_title_fixed,subcategory_title,article_count
0,Australia,Culture of Australia,Category:Culture of Australia,70
1,Australia,Politics of Australia,Category:Politics of Australia,57
2,Australia,Education in Australia,Category:Education in Australia,53
3,Australia,Government of Australia,Category:Government of Australia,37
4,Belgium,Culture of Belgium,Category:Culture of Belgium,41
5,Belgium,Politics of Belgium,Category:Politics of Belgium,40
6,Belgium,Government of Belgium,Category:Government of Belgium,35
7,Belgium,Economy of Belgium,Category:Economy of Belgium,22
8,Bosnia_and_Herzegovina,Culture of Bosnia and Herzegovina,Category:Culture of Bosnia and Herzegovina,46
9,Bosnia_and_Herzegovina,Bosnia and Herzegovina templates,Category:Bosnia and Herzegovina templates,40


np.int64(2211)

In [25]:
## Define a function that pulls four most popular articles from each country by ranking via pageviews. First, outline dates
start_date_2014 = 20140601
end_date_2014 = 20140730

start_date_2022 = 20221101
end_date_2022 = 20221231

start_date_2026 = 20260601
end_date_2026 = 20260730

## Create dictionary of start and end dates for each country
cup_windows = {
    'Australia': (start_date_2026, end_date_2026),
    'Belgium': (start_date_2014, end_date_2014),  # or handle 2014/2022 separately if kept distinct
    'Belgium': (start_date_2026, end_date_2026),
    'Bosnia_and_Herzegovina': (start_date_2026, end_date_2026),
    'England': (start_date_2022, end_date_2022),
    'Germany': (start_date_2014, end_date_2014),
    'Ghana': (start_date_2014, end_date_2014),
    'Iran': (start_date_2022, end_date_2022),
    'Netherlands': (start_date_2022, end_date_2022),
    'Paraguay': (start_date_2026, end_date_2026),
    'Portugal': (start_date_2014, end_date_2014),
    'Turkey': (start_date_2026, end_date_2026),
    'Wales': (start_date_2022, end_date_2022),
}

In [40]:
## Now pull article data for each subcategory, long run time.
article_rows = []

def articles_for_subcats(subcat_df_final):
    for _, row in subcat_df_final.iterrows():
        country = row['country']
        subcat_title = row['subcategory_title'] ## pw needs
        subcat = pw.Category(site, subcat_title)
        
        try:
            for page in subcat.articles():
                article_rows.append({
                    'country': country,
                    'subcategory_title_fixed': row['subcategory_title_fixed'],
                    'article_title': page.title()
                    'viewcount': page.
                })
            time.sleep(4)  
        except Exception as e:
            print(f"Error pulling articles from {subcat_title}: {e}")

articles_for_subcats(subcat_df_final)
articles_df = pd.DataFrame(article_rows)

https://en.wikipedia.org/w/api.php
The server may be down.
Status code: 429
User agent: /ipykernel_launcher (wikipedia:en) Pywikibot/11.6.0 (-1 (unknown)) Python/3.13.9.final.0 requests/2.32.5

The text message is:
You are making too many requests to the API.
Please follow the best practices at .
If you are unsure how to get the access you need, contact .
request-id: 372eac47-bd98-4899-a91c-701e492329ab

Set gcmlimit = ['250']
Sleeping for 10.9 seconds, 2026-08-15 12:32:48
https://en.wikipedia.org/w/api.php
The server may be down.
Status code: 429
User agent: /ipykernel_launcher (wikipedia:en) Pywikibot/11.6.0 (-1 (unknown)) Python/3.13.9.final.0 requests/2.32.5

The text message is:
You are making too many requests to the API.
Please follow the best practices at .
If you are unsure how to get the access you need, contact .
request-id: 70f759cc-7903-47e9-9b31-5d892bd0173b

Set gcmlimit = ['250']
Sleeping for 15.9 seconds, 2026-08-15 12:33:44
https://en.wikipedia.org/w/api.php
The serve

In [ ]:

headers = {
    'User-Agent': 'EskildsenLogan research project (logan.j.eskildsen.28@dartmouth.edu)'
}

def get_daily_pageviews(articles_df, cup_windows, headers):
    all_daily_records = []
    
    for _, row in articles_df.iterrows():
        country = row['country']
        subcat = row['subcategory_title_fixed']
        article = row['article_title'].replace(' ', '_')
        
        if country not in cup_windows:
            print(f"No date range defined for {country}, skipping {article}")
            continue
        start_date, end_date = cup_windows[country]
        
        pv_query = (
            'https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/'
            f'en.wikipedia.org/all-access/all-agents/{article}/daily/{start_date}/{end_date}'
        )
        
        try:
            pv_resp = requests.get(pv_query, headers=headers)
            if pv_resp.status_code == 200:
                items = pv_resp.json().get('items', [])
                for item in items:
                    all_daily_records.append({
                        'country': country,
                        'subcategory_title_fixed': subcat,
                        'article_title': article,
                        'date': item['timestamp'][:8],  # YYYYMMDD
                        'views': item.get('views', 0)
                    })
            else:
                # article had no pageview data in this window (common for 404s)
                all_daily_records.append({
                    'country': country,
                    'subcategory_title_fixed': subcat,
                    'article_title': article,
                    'date': None,
                    'views': 0
                })
        except Exception as e:
            print(f"Error pulling pageviews for {article}: {e}")
        
        time.sleep(0.2)  # lighter delay — this is a different, less strict API than en.wikipedia.org/w/api.php
    
    return pd.DataFrame(all_daily_records)

daily_views_df = get_daily_pageviews(articles_df, cup_windows, headers)
daily_views_df